# Solutions · 06 · Multiple reactions and selectivity

Worked solutions to the exercises of [notebook 06](../notebooks/06_multiple_reactions_and_selectivity.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engreact is missing): install it from GitHub
    import engreact
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engreact"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import integrate, optimize
from engreact import kinetics, multiple
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

For A -> B -> C with k1 = 0.5 and k2 = 0.2 1/min, find the CSTR residence time and the PFR space time that each give 90 % conversion of A, and compare the yields of B. Which reactor would you choose if B is the product, and if C is?

In [2]:
k1, k2 = 0.5, 0.2
tau_c = optimize.brentq(lambda tau: k1 * tau / (1 + k1 * tau) - 0.9, 0, 1e3)
tau_p = -np.log(0.1) / k1
Bc = float(multiple.series_cstr(k1, k2, 1.0, tau_c)["B"])
Bp = float(multiple.series_batch(k1, k2, 1.0, tau_p)["B"])
print(f"90 % conversion of A: CSTR tau = {tau_c:.1f} min, yield of B = {Bc/0.9:.3f}; PFR tau = {tau_p:.2f} min, yield of B = {Bp/0.9:.3f}")
print(f"C formed: CSTR {0.9 - Bc:.3f}, PFR {0.9 - Bp:.3f}")

90 % conversion of A: CSTR tau = 18.0 min, yield of B = 0.217; PFR tau = 4.61 min, yield of B = 0.552
C formed: CSTR 0.704, PFR 0.403


At the same 90 % conversion the PFR keeps 45 % of the reacted A as B, the CSTR only 20 %, because the CSTR
holds B at the outlet concentration for eighteen minutes. If B is the product, PFR; if C is, the CSTR gets
there with less volume than a PFR run long enough to consume B.

## Exercise 2

Two parallel reactions A -> D (first order, Ea = 40 kJ/mol) and A -> U (second order, Ea = 80 kJ/mol). Using `Network`, find the combination of temperature (300-400 K) and feed concentration (0.1-10 mol/L) that maximises the selectivity D/U in a PFR at 90 % conversion.

In [3]:
best = None
for T in np.linspace(300, 400, 11):
    for C0 in np.logspace(-1, 1, 9):
        kD, kU = kinetics.arrhenius(1e5, 40e3, T), kinetics.arrhenius(1e10, 80e3, T)
        net = multiple.Network([({"A": -1, "D": 1}, lambda C, kD=kD: kD * C["A"]), ({"A": -1, "U": 1}, lambda C, kU=kU: kU * C["A"] ** 2)])
        tau = optimize.brentq(lambda tau: 1 - net.pfr({"A": C0}, tau, n=50)["A"][-1] / C0 - 0.9, 1e-3, 1e5)
        out = net.pfr({"A": C0}, tau, n=50)
        S = out["D"][-1] / max(out["U"][-1], 1e-12)
        if best is None or S > best[0]:
            best = (S, T, C0, tau)
print(f"best selectivity D/U = {best[0]:.1f} at T = {best[1]:.0f} K, C_A0 = {best[2]:.2f} mol/L (tau = {best[3]:.1f})")

best selectivity D/U = 1675.6 at T = 300 K, C_A0 = 0.10 mol/L (tau = 212.1)


The desired reaction is first order with the lower activation energy, the undesired second order with the
higher: low concentration and low temperature both favour D, so the optimum sits at the corner of the
box (300 K, 0.1 mol/L). The price is a long residence time - selectivity and throughput pull opposite
ways, and the economics decide.

## Exercise 3

**Implement it yourself:** solve A -> B -> C in a batch reactor with solve_ivp and locate the time of maximum B numerically; compare with `multiple.series_optimum_time`.

In [4]:
sol = integrate.solve_ivp(lambda t, y: [-k1 * y[0], k1 * y[0] - k2 * y[1]], (0, 20), [1.0, 0.0], dense_output=True, rtol=1e-10)
t_max = optimize.minimize_scalar(lambda t: -sol.sol(t)[1], bounds=(0, 20), method="bounded").x
print(f"numerical t_max = {t_max:.4f} min; analytic ln(k2/k1)/(k2 - k1) = {multiple.series_optimum_time(k1, k2):.4f}")

numerical t_max = 3.0543 min; analytic ln(k2/k1)/(k2 - k1) = 3.0543


Two ODEs, a dense solution and a one-dimensional maximisation reproduce the analytic optimum to four
decimals - the numerical route that works when the kinetics are not first order and no formula exists.